# Delivery · 03 Gold
Owned by: Project planning. Reads from other domains only through shortcuts.

In [ ]:
# Domain: delivery · owner: Project planning
# Attach this notebook to the lakehouse "lh_delivery" in the workspace "Solvi Delivery".
DOMAIN = "delivery"
LANDING_PATH = f"Files/landing/{DOMAIN}"   # upload this domain's CSV files here
TABLE_FORMAT = "delta"
LOCAL_PREFIX = ""                            # local testing sets this to f"{DOMAIN}_"
AS_OF = "2026-10-31"                         # demo data ends here; use the current date with live data

from pyspark.sql import functions as F, Window as W

def write_table(df, name):
    (df.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable(LOCAL_PREFIX + name))
    print(f"wrote {name:<30} {df.count():>9,} rows")

def read_table(name):                         # a table this domain owns
    return spark.read.table(LOCAL_PREFIX + name)

def read_shared(domain, name):                # another domain's table, through a OneLake shortcut named <domain>_<name>
    return spark.read.table(f"{domain}_{name}")

Crews and jobs. Certification validity is checked against the job date here, where the crew data is owned.

In [ ]:
crew, cert, job = read_table("silver_crew"), read_table("silver_certification"), read_table("silver_job")
write_table(crew, "dim_crew")
certified = (job.alias("j").join(cert.alias("c"), (F.col("j.crew_id") == F.col("c.crew_id")) &
             F.col("j.job_date").between(F.col("c.valid_from"), F.col("c.valid_to")), "left")
             .groupBy("j.job_id").agg((F.count("c.crew_id") > 0).alias("crew_certified")))
prior = W.partitionBy("crew_id").orderBy("job_date", "job_id")
fact_job = (job.join(certified, "job_id")
  .withColumn("crew_prior_jobs", F.row_number().over(prior) - 1)
  .withColumn("days_late", F.datediff("job_date", "planned_date")))
write_table(fact_job, "fact_job")